# Settings

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR = "/content/drive/MyDrive/Islamic_Stories_Project"

LORA_PATH = f"{BASE_DIR}/models/Qwen2_5_7B_Arabic_Stories"
CSV_PATH = f"{BASE_DIR}/Qwen_output.csv"


for label, p in [
    ("LORA", LORA_PATH),
    ("Output CSV", CSV_PATH),
]:
    print(("✓" if os.path.exists(p) else "✗ MISSING"), label, "->", p)

Mounted at /content/drive
✓ LORA -> /content/drive/MyDrive/Islamic_Stories_Project/models/Qwen2_5_7B_Arabic_Stories
✗ MISSING Output CSV -> /content/drive/MyDrive/Islamic_Stories_Project/Qwen_output.csv


In [1]:
!pip install -q -U transformers accelerate peft bitsandbytes sentence-transformers pandas==2.2.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 157.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.9/588.9 kB 51.6 MB/s eta 0:00:00


# Load Qwen

In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4-bit QLoRA config — identical to your original notebook
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Base model
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    trust_remote_code=True,
    device_map="auto",
)

# Attach your trained LoRA adapter (no retraining)
ft_model = PeftModel.from_pretrained(base, LORA_PATH).eval()
print("✓ Qwen + LoRA loaded")

Device: cuda


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

✓ Qwen + LoRA loaded


# Functions

In [5]:
# ============================================================
# Functions — Qwen generation + summary + query + CSV saving
# ============================================================

import os
import csv
from datetime import datetime

# نفس system prompt القديم
SYSTEM_PROMPT = "أنت كوين، كاتب قصص عربية للأطفال يركز على القيم الإسلامية والعبرة في النهاية."


# ============================================================
# 1. Story generation
# ============================================================

def generate_story_qwen(
    age,
    moral,
    topic,
    place=None,
    end_of_story=None,
    dialogue=None,
    num_characters=None,
    country=None,
    season=None,
    activity=None,
    emotion=None,
    plot_twist=None,
    max_new_tokens=400
):
    """
    Generate Arabic children's story using the loaded ALLaM + LoRA model.
    Uses the same original prompt and generation settings.
    """

    features = [
        f"- عُمر الطفل/الطفلة: {age} سنة",
        f"- القيمة الإسلامية (العبرة): {moral}",
        f"- الموضوع العام للقصة: {topic}",
    ]

    if place:
        features.append(f"- مكان أحداث القصة: {place}")
    if country:
        features.append(f"- الدولة: {country}")
    if season:
        features.append(f"- الفصل: {season}")
    if activity:
        features.append(f"- النشاط الرئيسي في القصة: {activity}")
    if num_characters:
        features.append(f"- عدد الشخصيات الأساسية: {num_characters}")
    if emotion:
        features.append(f"- الشعور العام في القصة: {emotion}")

    if dialogue is not None:
        features.append(
            "- تضمين حوار بين الشخصيات"
            if dialogue
            else "- تقليل الحوار والتركيز على السرد"
        )

    if plot_twist is not None:
        features.append(
            "- تحتوي على حبكة مفاجِئة في النهاية"
            if plot_twist
            else "- بدون حبكة مفاجِئة"
        )

    if end_of_story:
        features.append(f"- شكل نهاية القصة المطلوب: {end_of_story}")

    user_prompt = (
        "أريد منك أن تكتب قصة عربية للأطفال بناءً على المواصفات التالية:\n\n"
        + "\n".join(features)
        + "\n\nشروط مهمة:\n"
        "- استخدم لغة عربية مبسطة وممتعة تناسب الأطفال.\n"
        "- اجعل القصة مترابطة وواضحة.\n"
        "- في النهاية، اكتب سطرًا يبدأ بكلمة: \"العبرة:\" ثم قدّم العبرة بشكل صريح وواضح."
    )

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_prompt
        },
    ]

    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(chat_text, return_tensors="pt").to(device)

    with torch.no_grad():
        out = ft_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=0.9,
            temperature=0.8,
        )

    gen_ids = out[0][inputs["input_ids"].shape[-1]:]
    story = tokenizer.decode(gen_ids, skip_special_tokens=True)

    return story.strip()


# ============================================================
# 2. Extract moral
# ============================================================

def extract_moral(text):
    """
    Extract only the explicit moral line after 'العبرة:'.
    """
    marker = "العبرة:"

    if marker not in text:
        return ""

    after_marker = text.split(marker, 1)[1].strip()
    lines = [line.strip() for line in after_marker.splitlines() if line.strip()]

    if not lines:
        return ""

    moral_line = lines[0]
    moral_line = moral_line.strip().strip('"').strip("“”").strip("«»")

    return moral_line.strip()


# ============================================================
# 3. Summary for retrieval
# ============================================================

def summarize_story_for_retrieval(story_text, topic, moral):
    """
    Generate short value-focused summary using ALLaM.
    Same summary prompt and settings.
    """

    prompt = f"""
لديك قصة عربية للأطفال، وأريد تلخيصها لغرض البحث عن حديث نبوي مناسب.

الموضوع: {topic}
القيمة الإسلامية التي أدخلها المستخدم: {moral}

القصة:
{story_text}

اكتب ملخصًا قصيرًا جدًا من جملة واحدة فقط.
ركّز على:
- الحدث الأساسي في القصة
- القيمة الإسلامية أو الأخلاقية
- العبرة المناسبة

تجنب ذكر التفاصيل الجانبية مثل الأسماء، المكان، الفصل، أو الوصف الطويل.
لا تكتب عنوانًا ولا شرحًا. اكتب الملخص فقط.
""".strip()

    messages = [
        {
            "role": "system",
            "content": "أنت مساعد يلخص قصص الأطفال العربية لغرض مطابقة القصة مع حديث نبوي مناسب."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(chat_text, return_tensors="pt").to(device)

    with torch.no_grad():
        out = ft_model.generate(
            **inputs,
            max_new_tokens=80,
            do_sample=False
        )

    gen_ids = out[0][inputs["input_ids"].shape[-1]:]
    summary = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

    summary = summary.replace("الملخص:", "").replace("ملخص:", "").strip()
    summary = summary.strip().strip('"').strip("“”").strip("«»")

    lines = [line.strip() for line in summary.splitlines() if line.strip()]
    if lines:
        summary = lines[0]

    return summary.strip()


# ============================================================
# 4. Query builders
# ============================================================

def build_query_with_summary(topic, moral, story_summary=None):
    """
    Query with summary.
    The summary is stored inside the query only.
    """
    q = (
        f"القيمة الإسلامية: {topic}\n"
        f"العبرة من القصة: {moral}"
    )

    if story_summary:
        q += f"\nملخص القصة: {story_summary}"

    return q


def build_query_no_summary(topic, moral):
    """
    Query without summary.
    """
    return (
        f"القيمة الإسلامية: {topic}\n"
        f"العبرة من القصة: {moral}"
    )


# ============================================================
# 5. CSV saving
# ============================================================

GEN_COLUMNS = [
    "sample_id",
    "timestamp",

    "model_name",

    "input_age",
    "input_topic",
    "input_moral",
    "input_place",
    "input_country",
    "input_season",
    "input_activity",
    "input_emotion",
    "input_dialogue",
    "input_plot_twist",
    "input_end_of_story",

    "generated_story",
    "extracted_moral",

    "query_with_summary",
    "query_no_summary",
]


def append_generation_csv(path, row):
    """
    Append one generation result to CSV.
    """
    new_file = not os.path.exists(path)

    parent = os.path.dirname(os.path.abspath(path))
    if parent:
        os.makedirs(parent, exist_ok=True)

    with open(path, "w" if new_file else "a", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=GEN_COLUMNS)

        if new_file:
            writer.writeheader()

        writer.writerow(row)

    return row


def get_next_sample_id(path):
    """
    Auto sample_id based on existing CSV rows.
    """
    if os.path.exists(path):
        try:
            old_df = pd.read_csv(path)
            return len(old_df) + 1
        except Exception:
            return 1

    return 1


print("✓ Qwen generation functions ready")

✓ Qwen generation functions ready


# أمثلة

مثال الصدق

In [6]:
# ============================================================
# Run Qwen generation + summary + queries + save
# ============================================================

sample_id = get_next_sample_id(CSV_PATH)

inp = {
    "age": 8,
    "topic": "الصدق",
    "moral": "أهمية الصدق",
    "place": "المدرسة",
    "country": "السعودية",
    "season": "الخريف",
    "activity": "وقت اللعب",
    "emotion": "حماسي",
    "dialogue": True,
    "plot_twist": True,
    "end_of_story": "اعتراف صادق",
    "num_characters": None,
}

print("=" * 80)
print("Running Qwen generation")
print("Sample ID:", sample_id)
print("=" * 80)

story = generate_story_qwen(
    age=inp["age"],
    moral=inp["moral"],
    topic=inp["topic"],
    place=inp.get("place"),
    country=inp.get("country"),
    season=inp.get("season"),
    activity=inp.get("activity"),
    emotion=inp.get("emotion"),
    dialogue=inp.get("dialogue"),
    plot_twist=inp.get("plot_twist"),
    end_of_story=inp.get("end_of_story"),
    num_characters=inp.get("num_characters"),
    max_new_tokens=400,
)

extracted_moral = extract_moral(story)

story_summary = summarize_story_for_retrieval(
    story_text=story,
    topic=inp["topic"],
    moral=inp["moral"]
)

query_with_summary = build_query_with_summary(
    topic=inp["topic"],
    moral=inp["moral"],
    story_summary=story_summary
)

query_no_summary = build_query_no_summary(
    topic=inp["topic"],
    moral=inp["moral"]
)

out_row = {
    "sample_id": sample_id,
    "timestamp": datetime.now().isoformat(timespec="seconds"),

    "model_name": "Qwen",

    "input_age": inp.get("age"),
    "input_topic": inp.get("topic"),
    "input_moral": inp.get("moral"),
    "input_place": inp.get("place"),
    "input_country": inp.get("country"),
    "input_season": inp.get("season"),
    "input_activity": inp.get("activity"),
    "input_emotion": inp.get("emotion"),
    "input_dialogue": inp.get("dialogue"),
    "input_plot_twist": inp.get("plot_twist"),
    "input_end_of_story": inp.get("end_of_story"),

    "generated_story": story,
    "extracted_moral": extracted_moral,

    "query_with_summary": query_with_summary,
    "query_no_summary": query_no_summary,
}

append_generation_csv(CSV_PATH, out_row)

print("\n" + "=" * 80)
print("Generated Story")
print("=" * 80)
print(story)

print("\n" + "=" * 80)
print("Extracted Moral")
print("=" * 80)
print(extracted_moral)

print("\n" + "=" * 80)
print("Query WITH summary")
print("=" * 80)
print(query_with_summary)

print("\n" + "=" * 80)
print("Query WITHOUT summary")
print("=" * 80)
print(query_no_summary)

print("\nSaved to:")
print(CSV_PATH)

Running Qwen generation
Sample ID: 1

Generated Story
في يوم خريف جميل في مدينة الرياض بالمملكة العربية السعودية، كان الأطفال في مدرستهم يلعبون معًا تحت ظلال الأشجار الكثيفة التي تنضح بالجمال. كانت جوهرة طفلة صغيرة عمرها ثمانية سنوات تشارك في اللعب مع زميلاتها في الصف.

قالت جوهرة لزميلاتها: "يا خواتي، نظفت قلمي الجديد وأود أن نلعب به بعد الدرس، لنكون أول من يستخدمه."

قالت زميلتها سارة: "حسنا يا جوهرة، لكن لا ننسى أن نقوم بتنظيف قلمنا أيضًا بعد استخدامه."

قالت جوهرة بنصيحة: "طبعاً سارا، سنقوم بذلك بعد الدرس."

لكن في اللحظة الأخيرة، قبل البدء باللعب، ذكرت جوهرة للجميع أن القلم لم يُستخدم بعد. فنظرت إليها زميلاتها باندهاش، وقالت سارة: "جوهرة، هل تقول إنك لم تستخدم القلم الجديد؟"

قالت جوهرة: "نعم، كنت صادقة معكم. لم أستخدم القلم الجديد لأنه لم يكن محتاجًا."

قالت سارة بنصيحة: "جوهرة، هذه هي الأخلاق الحسنة. الصدق هو مفتاح الحفاظ على الثقة."

ثم قالت جوهرة: "شكراً سارة، سأتعلم من هذا اليوم."

لكن في النهاية، حدث شيء مفاجئ. عندما فتحت جوهرة قلمها وحاولت استخدامه، لاحظت أنها فاتت الكتابة 

مثال الصيام

In [10]:
# ============================================================
# Run Qwen generation + summary + queries + save
# ============================================================

sample_id = get_next_sample_id(CSV_PATH)

inp = {
    "age": 8,
    "topic": "الصيام",
    "moral": "فضل الصيام",
    "place": "المنزل",
    "country": "السعودية",
    "season": "الشتاء",
    "activity": "الاستعداد لشهر رمضان",
    "emotion": "روحاني",
    "dialogue": True,
    "plot_twist": False,
    "end_of_story": "طمأنينة",
    "num_characters": None,
}

print("=" * 80)
print("Running Qwen generation")
print("Sample ID:", sample_id)
print("=" * 80)

story = generate_story_qwen(
    age=inp["age"],
    moral=inp["moral"],
    topic=inp["topic"],
    place=inp.get("place"),
    country=inp.get("country"),
    season=inp.get("season"),
    activity=inp.get("activity"),
    emotion=inp.get("emotion"),
    dialogue=inp.get("dialogue"),
    plot_twist=inp.get("plot_twist"),
    end_of_story=inp.get("end_of_story"),
    num_characters=inp.get("num_characters"),
    max_new_tokens=400,
)

extracted_moral = extract_moral(story)

story_summary = summarize_story_for_retrieval(
    story_text=story,
    topic=inp["topic"],
    moral=inp["moral"]
)

query_with_summary = build_query_with_summary(
    topic=inp["topic"],
    moral=inp["moral"],
    story_summary=story_summary
)

query_no_summary = build_query_no_summary(
    topic=inp["topic"],
    moral=inp["moral"]
)

out_row = {
    "sample_id": sample_id,
    "timestamp": datetime.now().isoformat(timespec="seconds"),

    "model_name": "Qwen",

    "input_age": inp.get("age"),
    "input_topic": inp.get("topic"),
    "input_moral": inp.get("moral"),
    "input_place": inp.get("place"),
    "input_country": inp.get("country"),
    "input_season": inp.get("season"),
    "input_activity": inp.get("activity"),
    "input_emotion": inp.get("emotion"),
    "input_dialogue": inp.get("dialogue"),
    "input_plot_twist": inp.get("plot_twist"),
    "input_end_of_story": inp.get("end_of_story"),

    "generated_story": story,
    "extracted_moral": extracted_moral,

    "query_with_summary": query_with_summary,
    "query_no_summary": query_no_summary,
}

append_generation_csv(CSV_PATH, out_row)

print("\n" + "=" * 80)
print("Generated Story")
print("=" * 80)
print(story)

print("\n" + "=" * 80)
print("Extracted Moral")
print("=" * 80)
print(extracted_moral)

print("\n" + "=" * 80)
print("Query WITH summary")
print("=" * 80)
print(query_with_summary)

print("\n" + "=" * 80)
print("Query WITHOUT summary")
print("=" * 80)
print(query_no_summary)

print("\nSaved to:")
print(CSV_PATH)

Running Qwen generation
Sample ID: 2

Generated Story
في بلدنا الجميل المملكة العربية السعودية، كان هناك منزل صغير يعيش فيه شقيقان، أحمد وأمنا. كان شهر رمضان قادماً، والأشهر القارس شتاءً يحمل معه البرد والثلج. كان أحمد يشعر بالسعادة لأن شقيقته أمنا ستكون معهم لتشاركهم هذا الشهر المبارك.

كان أحمد يحب الصيام، فقد كان يرى فيه فضيلة ومحبة لله سبحانه وتعالى. أما أمنا، فهي كانت تحب الصيام أيضاً، ولكنها لم تكن تفهم تماماً لماذا يصوم الناس. يوماً، سألته أمنا: "أخي أحمد، لماذا نصوم في رمضان؟ لماذا لا نأكل ولا نشرب في هذا الشهر؟"

قال أحمد بحنان: "لأننا نصوم لأجل الله عز وجل. نصوم لنذكر نفسنا بأننا محظوظون وأن لدينا الكثير من الطعام والشراب. كما أنه يعلمنا قيمة الطعام ونعلم أن نكون شاكرين لما نملك".

قالت أمنا بصوت هادئ: "فكرة جميلة يا أحمد. لكنني لا أستطيع الصيام. أنا أحتاج إلى الطعام والشراب في هذا البرد القارس. هل يمكنني أن أساعدكم بطريقة أخرى؟"

رد أحمد بفرح: "نعم يا أمنا، يمكنك أن تساعدنا بتحضير الأطعمة الصحية والطيبة لنا في رمضان. هذا سيجعلنا نشعر بالأمان والراحة."

بدأ الشقيقان في الاستع

مثال الصلاة

In [12]:
# ============================================================
# Run Qwen generation + summary + queries + save
# ============================================================

sample_id = get_next_sample_id(CSV_PATH)

inp = {
    "age": 8,
    "topic": "الصلاة",
    "moral": "أهمية الصلاة",
    "place": "المسجد",
    "country": "السعودية",
    "season": "الصيف",
    "activity": "الذهاب للمسجد",
    "emotion": "حنون",
    "dialogue": True,
    "plot_twist": False,
    "end_of_story": "نهاية سعيدة",
    "num_characters": None,
}

print("=" * 80)
print("Running Qwen generation")
print("Sample ID:", sample_id)
print("=" * 80)

story = generate_story_qwen(
    age=inp["age"],
    moral=inp["moral"],
    topic=inp["topic"],
    place=inp.get("place"),
    country=inp.get("country"),
    season=inp.get("season"),
    activity=inp.get("activity"),
    emotion=inp.get("emotion"),
    dialogue=inp.get("dialogue"),
    plot_twist=inp.get("plot_twist"),
    end_of_story=inp.get("end_of_story"),
    num_characters=inp.get("num_characters"),
    max_new_tokens=400,
)

extracted_moral = extract_moral(story)

story_summary = summarize_story_for_retrieval(
    story_text=story,
    topic=inp["topic"],
    moral=inp["moral"]
)

query_with_summary = build_query_with_summary(
    topic=inp["topic"],
    moral=inp["moral"],
    story_summary=story_summary
)

query_no_summary = build_query_no_summary(
    topic=inp["topic"],
    moral=inp["moral"]
)

out_row = {
    "sample_id": sample_id,
    "timestamp": datetime.now().isoformat(timespec="seconds"),

    "model_name": "Qwen",

    "input_age": inp.get("age"),
    "input_topic": inp.get("topic"),
    "input_moral": inp.get("moral"),
    "input_place": inp.get("place"),
    "input_country": inp.get("country"),
    "input_season": inp.get("season"),
    "input_activity": inp.get("activity"),
    "input_emotion": inp.get("emotion"),
    "input_dialogue": inp.get("dialogue"),
    "input_plot_twist": inp.get("plot_twist"),
    "input_end_of_story": inp.get("end_of_story"),

    "generated_story": story,
    "extracted_moral": extracted_moral,

    "query_with_summary": query_with_summary,
    "query_no_summary": query_no_summary,
}

append_generation_csv(CSV_PATH, out_row)

print("\n" + "=" * 80)
print("Generated Story")
print("=" * 80)
print(story)

print("\n" + "=" * 80)
print("Extracted Moral")
print("=" * 80)
print(extracted_moral)

print("\n" + "=" * 80)
print("Query WITH summary")
print("=" * 80)
print(query_with_summary)

print("\n" + "=" * 80)
print("Query WITHOUT summary")
print("=" * 80)
print(query_no_summary)

print("\nSaved to:")
print(CSV_PATH)

Running Qwen generation
Sample ID: 3

Generated Story
في قلب مدينة مشرقة بسحرها وجمالها، كان يعيش صغيراً يدعى حسن، عمره الثمانية. كان حسن محبوباً بين أقرانه ويتقن كل ما يتعلق بالرياضات واللعب في فصل الصيف. لكن، كان له هواية خاصة تأتي من قلبه، وهو زيارة المسجد في صلاة الفجر والأمساك.

كان يوماً عاد حسن كالمعتاد إلى المسجد، ليجد الأبواب مغلقة. قلقاً، سأل جيرانه عن السبب، فأجابوا أنه لا يوجد خطبة اليوم. بعدها، بدأ حسن يتأمل في نفسه، وبدأ يسأل نفسه، "لماذا أذهب إلى المسجد؟" فوجد الإجابة أنه يبحث عن السلام في قلبه وروحه.

خلال رحلته الصيفية، التقط حسن بعض الأزهار التي كانت تنمو تحت أشجار المسجد ووضعها في زاوية صغيرة تحت صندوق الصلاة، مضيفاً قلادة صغيرة كتبت عليها كلماته المحبة لأبيه. وفي أحد الأيام، كان حسن في طريقه إلى المسجد عندما لاحظ أن الأبواب قد فتحت، فقرر أن ينتظر حتى يرى خطيباً يتحدث عن أهمية الصلاة في حياتنا.

قال الخطيب بصوت واضح: "يا أبناء المؤمنين، الصلاة هي ربطنا بأعين الله، وهي طريقة نظهر فيها إخلاصنا له. كل يوم نصلي، نستذكر أننا تحت نظر الله وأن هناك علاقات وهمية بيننا وبينه.

In [13]:
import os
import pandas as pd

print("CSV_PATH:", CSV_PATH)
print("Exists:", os.path.exists(CSV_PATH))

if os.path.exists(CSV_PATH):
    df = pd.read_csv(CSV_PATH)
    print("Rows:", len(df))
    display(df.tail())
else:
    print("File does not exist yet.")

CSV_PATH: /content/drive/MyDrive/Islamic_Stories_Project/Qwen_output.csv
Exists: True
Rows: 3


,sample_id,timestamp,model_name,input_age,input_topic,input_moral,input_place,input_country,input_season,input_activity,input_emotion,input_dialogue,input_plot_twist,input_end_of_story,generated_story,extracted_moral,query_with_summary,query_no_summary
0,1,2026-05-22T05:07:49,Qwen,8,الصدق,أهمية الصدق,المدرسة,السعودية,الخريف,وقت اللعب,حماسي,True,True,اعتراف صادق,في يوم خريف جميل في مدينة الرياض بالمملكة العر...,NaN,القيمة الإسلامية: الصدق\nالعبرة من القصة: أهمي...,القيمة الإسلامية: الصدق\nالعبرة من القصة: أهمي...
1,2,2026-05-22T05:09:55,Qwen,8,الصيام,فضل الصيام,المنزل,السعودية,الشتاء,الاستعداد لشهر رمضان,روحاني,True,False,طمأنينة,في بلدنا الجميل المملكة العربية السعودية، كان ...,NaN,القيمة الإسلامية: الصيام\nالعبرة من القصة: فضل...,القيمة الإسلامية: الصيام\nالعبرة من القصة: فضل...
2,3,2026-05-22T05:11:02,Qwen,8,الصلاة,أهمية الصلاة,المسجد,السعودية,الصيف,الذهاب للمسجد,حنون,True,False,نهاية سعيدة,في قلب مدينة مشرقة بسحرها وجمالها، كان يعيش صغ...,NaN,القيمة الإسلامية: الصلاة\nالعبرة من القصة: أهم...,القيمة الإسلامية: الصلاة\nالعبرة من القصة: أهم...
